> ## XAI 기반 OTT 신규 고객 이탈 요인 분석 및 리텐션 전략

OTT 이탈 예측 프로젝트의 전체 계획 문서.
이 노트북은 분석을 직접 수행하는 것이 아니라, 이후 `01`번부터 `10`번까지의 노트북이 어떤 문제의식, 어떤 데이터 기준, 어떤 전처리 정책, 어떤 해석 원칙 위에서 작성되어야 하는지를 고정하기 위해 작성되었음.

처음 보는 팀원이나 새 대화의 LLM이 이 파일만 읽어도 아래 질문에 답할 수 있어야 함을 목적으로 두고 작성하려 노력하였음.

1. 이 프로젝트는 무엇을 하려는 프로젝트인가?
2. 왜 100원딜 고객을 핵심 분석 대상으로 삼는가?
3. 왜 가입 후 1~3주차만 관측하고 4주차는 피처로 쓰지 않는가?
4. 어떤 가설은 살아남았고, 어떤 가설은 약하거나 폐기되었는가?
5. 앞으로 어떤 노트북을 어떤 순서로 작성해야 하는가?
6. 어떤 표현은 쓰면 안 되고, 어떤 표현은 안전한가?

(이 파일은 프로젝트의 지도가 되어야 합니다. 마치 지도에서 산을 오르지는 않지만, 어느 능선을 탈지 정할 수 있듯.)

In [ ]:
from pathlib import Path

# 웬만하면 park.ingyeom 폴더에서 실행하는 것을 권장함
# 예: .../ott-churn-prediction/park.ingyeom

def find_project_root(start: Path | None = None) -> Path:
    
	# 현재 위치 또는 상위 폴더에서 park.ingyeom 프로젝트 루트를 찾기
    start = Path.cwd() if start is None else Path(start).resolve()
    candidates = [start, *start.parents]
    for path in candidates:
        if path.name == "park.ingyeom":
            return path
        if (path / "park.ingyeom").exists():
            return path / "park.ingyeom"
    return start

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "_data"
RAW_DATA_DIR = DATA_DIR / "01_raw"
INTERIM_DATA_DIR = DATA_DIR / "02_interim"
PROCESSED_DATA_DIR = DATA_DIR / "03_processed"
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR)
print("NOTEBOOKS_DIR:", NOTEBOOKS_DIR)

PROJECT_ROOT: c:\Code\ott-churn-prediction\park.ingyeom
DATA_DIR: c:\Code\ott-churn-prediction\park.ingyeom\_data
NOTEBOOKS_DIR: c:\Code\ott-churn-prediction\park.ingyeom\notebooks


> #### 0-2. 프로젝트 한 문장 정의

이 프로젝트의 한 줄 정의:  
 **OTT 서비스의 신규 구독자의 1~3주차 행동과 멤버십 정보를 바탕으로 재구독 여부를 예측하고,  
 4주차 대응기간에 개입할 고위험 고객 세그먼트를 도출하는 분석 프로젝트**.

멘토님과의 협의 후 도출된 과제 명 또한 다음과 같이 정의되었음.
> XAI 기반 OTT 신규 고객 이탈 요인 분석 및 리텐션 전략

이 과제명에서 중요한 단어는 4개.

| 키워드 | 의미 |
|---|---|
| XAI | 단순 예측 성능보다 설명 가능한 요인을 찾는 것이 중요함 |
| 신규 고객 | 장기 고객이 아니라 신규 유입 고객의 초기 행동을 봄 |
| 이탈 요인 | 재구독하지 않는 고객의 특성을 찾음 |
| 리텐션 전략 | 모델 결과를 실제 대응 전략으로 연결해야 함 |

> #### 0-3. 비즈니스 문제 정의

이 프로젝트의 실무 질문은 다음과 같이 정리합니다.

```text
100원딜로 유입된 신규 고객 중
누가 재구독하지 않을 가능성이 높은가?

그리고 이 고객들을 4주차 대응기간 안에
어떤 방식으로 붙잡을 수 있는가?
```

중요한 점은 단순히 재구독률 차이를 보고 끝내는 것이 아닙니다. 이 프로젝트는 다음 흐름을 가져야 합니다.

```text
1~3주차 행동 관측
→ 이탈 위험 고객 선별
→ 4주차 대응기간에 개입
→ 세그먼트별 리텐션 전략 제안
```

따라서 모델의 목적은 전체 고객을 완벽하게 맞히는 것이 아니라, 4주차에 조치할 만한 고객군을 설명 가능하게 선별하는 것입니다.

## 0-4. 관측창과 대응창

이 프로젝트에서 가장 중요한 설계는 관측창과 대응창의 분리입니다.

```text
가입 후 day 0~20: 관측창
가입 후 day 21~27: 대응창
구독 종료 후 또는 다음 결제 여부: 결과
```

여기서 관측창은 고객의 행동을 보는 기간입니다. 대응창은 이탈 위험 고객에게 쿠폰, 요금제 전환 제안, 콘텐츠 추천, 알림 등의 리텐션 액션을 수행할 수 있는 기간입니다.

따라서 4주차 행동 데이터는 예측 피처로 사용하지 않습니다.

이유는 다음과 같습니다.

1. 4주차는 실제 대응해야 하는 기간이므로, 피처로 쓰면 실무적으로 늦습니다.
2. 4주차 행동은 이탈 직전 행동에 가까워 누수에 해당할 수 있습니다.
3. 이 프로젝트의 목표는 사후 탐지가 아니라 조기 경고입니다.

정리하면 다음 문장이 핵심입니다.

```text
4주차는 중요하지 않아서 제외하는 것이 아니라,
가장 중요하기 때문에 대응기간으로 남겨둔다.
```

## 0-5. 핵심 분석 대상

전체 고객 분석도 수행하지만, 이 프로젝트의 중심 타겟은 **100원딜 고객**입니다.

현재 데이터에서 확인된 중요한 사실은 다음입니다.

```text
price == 100
is_promotion == 1
```

두 조건은 사실상 같은 고객군으로 확인되었습니다. 따라서 현재 분석에서는 아래와 같이 정의합니다.

```text
100원딜 고객 = 프로모션 고객 = is_promotion == 1 = price == 100
```

다만 표현할 때는 주의해야 합니다. 100원딜 고객의 재구독률이 낮다고 해서, 100원딜이 이탈을 인과적으로 유발했다고 말하면 안 됩니다. 프로모션은 무작위 배정된 실험이 아니므로, 100원딜 고객군 자체가 일반 고객군과 다를 수 있습니다.

안전한 표현은 다음입니다.

```text
100원딜 고객군은 일반 고객군과 다른 재구독 행동을 보였다.
100원딜 고객은 별도의 세그먼트로 분석할 필요가 있다.
```

## 0-6. 현재까지 확인된 핵심 사실

현재까지의 초벌 재현 분석에서 가장 중요한 사실은 다음입니다.

| 번호 | 확인된 사실 | 현재 해석 |
|----:|-----|-----|
| 1 | 100원딜 고객은 비100원딜 고객보다 재구독률이 낮음 | 100원딜 고객군은 일반 고객군과 다른 재구독 행동을 보였기 때문에 별도의 세그먼트로 분석한 필요가 있다 |
| 2 | 2인 요금제의 재구독률이 가장 높음 | 2인 요금제부터 TV로 시청이 가능하기 때문, 그리고 어느정도 적정한 가격의 균형이 잡히는 안정적 구간이어서? |
| 3 | 4인 요금제의 재구독률이 낮음 | 베이직(7,900원)에 비해 프리미엄(13,900원)은 거진 2배나 차이나는 가격이다. 그렇기에 고사양 요금제를 100원 딜로 체험은 했지만, 향후 정가로의 전환이 부담될 가능성이 존재함 |
| 4 | 100원딜 + 4인 요금제 고객은 특히 재구독률이 낮음 | 3번에서 언급한 이유에 의한 고위험 세그먼트 후보 |
| 5 | 총 시청시간, 총 세션 수, 고유 콘텐츠 수는 강하지 않음 | 많이 봤다는 사실만으로는 유지 여부를 설명하기 어려움 |
| 6 | 1주차 비중, 3주차 증가량, daily slope는 일부 신호가 있음 | 초반 루틴화와 후반 체험 소진을 구분할 가능성 |
| 7 | 장르 단독 효과는 약함 | 장르는 핵심 원인보다 세그먼트 해석 보조 변수 |
| 8 | 영화 메타데이터는 버릴 필요는 없음 | 관람등급, 국가, 러닝타임, 최신성 등 보조 피처로 활용 가능 |
| 9 | 40대 2인 인증 100원딜 키즈/애니 가설은 유의하지 않음 | 생활형 이용 해석의 보조 아이디어로만 유지 |
| 10 | 1차 모델 AUC는 높지 않음 | 정밀 예측보다 리스크 랭킹과 세그먼트 해석용으로 보는 것이 안전함 |

## 0-7. 살아남은 가설

현재까지의 분석에서 완전히 확정된 것은 아니지만, 데이터와 비즈니스 해석이 함께 살아남은 가설은 다음입니다.

### 가설 A. 100원딜 고객은 일반 고객과 다른 재구독 행동을 보인다

100원딜 고객은 일반 고객보다 재구독률이 낮게 나타났습니다. 따라서 전체 고객을 하나로 뭉쳐서 보는 것보다, 100원딜 고객을 별도 분석하는 것이 타당합니다.

### 가설 B. 100원딜 + 4인 요금제 고객은 고위험군이다

100원딜로 4인 요금제를 선택한 고객은 프로모션 기간 동안 고사양 요금제를 체험했지만, 정가 전환 시 가격 부담이 커져 이탈할 가능성이 있습니다.

### 가설 C. 100원딜 + 2인 요금제 고객은 상대적으로 안정적이다

2인 요금제는 TV 시청 가능성과 가격 부담 사이의 균형점일 수 있습니다. 식사 중 시청, 가족 동반 시청, 아이에게 틀어주는 생활형 이용으로 정착할 가능성이 있습니다.

### 가설 D. 초반 루틴화가 유지 가능성과 연결될 수 있다

초반부터 일정하게 시청한 고객은 서비스가 생활 루틴에 들어갔을 가능성이 있습니다.

### 가설 E. 후반부(3주차 즈음) 시청 증가 또는 체험 소진형 행동은 이탈 위험과 연결될 수 있다

3주차에 시청이 몰리는 고객은 프로모션 기간 안에 보고 싶은 콘텐츠를 소진한 뒤 이탈할 가능성이 있습니다.

## 0-8. 약하거나 폐기된 가설

아래 가설들은 완전히 무의미하다고 단정하지는 않지만, 현재 데이터에서는 중심 가설로 삼기 어렵습니다.

| 가설 | 현재 판단 | 이유 |
|---|---|---|
| 많이 보면 재구독한다 | 약함 | 총 시청시간, 총 세션 수, 고유 콘텐츠 수가 강하지 않았음 |
| 장르가 이탈을 직접 결정한다 | 약함 | top genre와 주요 장르 비율이 전체적으로 유의하지 않았음 |
| 40대 2인 인증 100원딜 키즈/애니 고객은 재구독률이 유의하게 높다 | 약함 | 해당 고객은 존재하지만 표본이 작고 차이가 유의하지 않았음 |
| 4주차 행동까지 넣으면 더 좋은 모델이므로 써야 한다 | 사용 금지 | 4주차는 대응기간이므로 피처로 쓰면 실무 목적과 어긋남 |
| 프로모션이 이탈을 유발했다 | 사용 금지 | 무작위 실험이 아니므로 인과 단정 불가 |

## 0-9. 분석 원칙

이 프로젝트에서 계속 지켜야 할 분석 원칙은 다음입니다.

### 원칙 1. 확정 사실과 해석을 분리한다

예를 들어 아래 문장은 확정 사실에 가깝습니다.

```text
100원딜 + 4인 요금제 고객의 재구독률은 낮게 나타났다.
```

반면 아래 문장은 해석입니다.

```text
정가 전환 부담 때문에 이탈했을 가능성이 있다.
```

보고서와 발표에서는 두 문장을 섞지 말아야 합니다.

### 원칙 2. 프로모션 효과를 인과적으로 단정하지 않는다

이 데이터는 A/B 테스트 데이터가 아닙니다. 따라서 프로모션 고객과 비프로모션 고객의 차이는 프로모션 자체의 인과효과가 아니라 고객군 차이일 수 있습니다.

### 원칙 3. 4주차 데이터를 피처로 쓰지 않는다

4주차는 대응기간입니다. 4주차 행동을 피처로 쓰면 예측이 아니라 사후 탐지에 가까워질 수 있습니다.

### 원칙 4. 시청량보다 시청 패턴을 중시한다

현재까지 단순 시청량 변수는 강하지 않았습니다. 초반 루틴화, 후반 증가, 몰아보기, active span, watch slope 같은 패턴 변수를 더 중요하게 봅니다.

### 원칙 5. 콘텐츠 메타데이터는 보조축으로 둔다

영화 메타데이터는 버리지 않습니다. 다만 장르나 콘텐츠 성향만으로 이탈을 설명하려고 하지 않습니다. 요금제와 시청 패턴으로 만든 세그먼트의 성격을 설명하는 데 활용합니다.

### 원칙 6. 모델 성능을 과장하지 않는다

1차 모델 AUC가 높지 않았으므로, 이 모델을 정밀 예측 시스템처럼 말하면 안 됩니다. 리스크 랭킹과 세그먼트 탐색용 도구로 보는 것이 안전합니다.

## 0-10. 데이터 파일과 역할

이 프로젝트에서 사용하는 핵심 데이터는 다음과 같습니다.

| 파일 | 역할 |
|---|---|
| `Membership_v1.csv` | 종속변수 `is_repurchase`, 프로모션 여부, 요금제, 연령 등 멤버십 정보 |
| `User_Mapping_v1.csv` | `USER_KEY`와 `USER_NUM` 연결 |
| `View_History_v1.csv` | 유저별 영화 시청 이력 |
| `Movie_Master_v1.csv` | `MOVIE_NUM`에 해당하는 영화 제목과 공개월 정보 |
| `wavve_movies_filtered_by 정규식.csv` | Wavve 크롤링 기반 영화 메타데이터 |
| `wavve_notfound_kobis_filtered_by_char_match.csv` | Wavve 미수집분에 대한 KOBIS 보완 메타데이터 |
| `movie_metadata_unified_v2.csv` | Movie_Master 기준으로 Wavve/KOBIS를 통합한 분석용 영화 메타데이터 |
| `modeling_feature_table_with_content.csv` | 행동 피처와 콘텐츠 피처를 결합한 모델링용 feature table |

데이터 폴더 기준은 다음과 같이 권장합니다.

```text
_data/
├─ 01_raw/        원본 데이터
├─ 02_interim/    중간 병합 또는 통합 데이터
└─ 03_processed/  모델링과 최종 분석에 사용할 처리 완료 데이터
```

## 0-11. 전처리 정책 요약

구체적인 전처리 코드는 `02_preprocessing_policy.ipynb`에서 작성합니다. 여기서는 정책만 고정합니다.

| 정책 | 현재 결정 |
|---|---|
| 더미 이상치 | `gender=N`, `is_user_verified=0`, `age=40` 동시 만족 케이스 제거 |
| 분석 단위 | 개인 생애 단위가 아니라 구독 이벤트 또는 멤버십 행 단위 |
| USER_KEY 중복 | 복수 계정 또는 재가입 가능성으로 해석. 무조건 오류로 보지 않음 |
| 100원딜 정의 | `price == 100` 또는 `is_promotion == 1` |
| 관측창 | 고객별 `reg_date` 기준 day 0~20 |
| 대응창 | day 21~27 |
| 시청이력 없음 | 버리지 않고 `no_watch_obs_flag` 등으로 표시 |
| KOBIS 저신뢰 매칭 | `movie_metadata_unified_v2`에서 low-confidence 처리 |
| 콘텐츠 피처 기준 | `use_for_content_features == 1`인 영화 메타데이터만 사용 |

## 0-12. 전체 노트북 파이프라인

앞으로 작성할 노트북은 아래 순서를 따릅니다.

| 번호 | 노트북 | 목적 | 주요 출력 |
|---:|---|---|---|
| 00 | `00_project_plan.ipynb` | 프로젝트 목적과 분석 원칙 고정 | 프로젝트 지도 |
| 01 | `01_data_overview.ipynb` | 원본 데이터 구조, 행 수, 키 관계 확인 | 데이터 개요표 |
| 02 | `02_preprocessing_policy.ipynb` | 전처리 정책 적용과 검산 | 전처리 완료 membership, 관측창 view |
| 03 | `03_movie_metadata_unification.ipynb` | Movie_Master 기준 Wavve/KOBIS 통합 | `movie_metadata_unified_v2.csv` |
| 04 | `04_usage_feature_engineering.ipynb` | View_History 기반 행동 피처 생성 | `user_usage_features.csv` |
| 05 | `05_content_feature_engineering.ipynb` | 영화 메타데이터 기반 콘텐츠 피처 생성 | `user_content_features.csv` |
| 06 | `06_significance_tests.ipynb` | 전체, 100원딜, 2인, 4인 집단별 검정 | 유의성 검정표 |
| 07 | `07_modeling_baseline.ipynb` | 1차 모델링과 성능 확인 | 모델 성능표, feature importance |
| 08 | `08_xai_shap_interpretation.ipynb` | SHAP 기반 XAI 분석 | SHAP summary, dependence plot |
| 09 | `09_segmentation_strategy.ipynb` | 세그먼트 정의와 전략 연결 | 세그먼트별 재구독률, 전략표 |
| 10 | `10_business_simulation_and_final_checks.ipynb` | 비즈니스 시뮬레이션과 최종 검산 | 기대효과표, 최종 숫자 |

In [ ]:
import pandas as pd

notebook_plan = pd.DataFrame([
    [0, "00_project_plan.ipynb", "프로젝트 목적과 분석 원칙 고정", "프로젝트 지도"],
    [1, "01_data_overview.ipynb", "원본 데이터 구조, 행 수, 키 관계 확인", "데이터 개요표"],
    [2, "02_preprocessing_policy.ipynb", "전처리 정책 적용과 검산", "전처리 완료 membership, 관측창 view"],
    [3, "03_movie_metadata_unification.ipynb", "Movie_Master 기준 Wavve/KOBIS 통합", "movie_metadata_unified_v2.csv"],
    [4, "04_usage_feature_engineering.ipynb", "View_History 기반 행동 피처 생성", "user_usage_features.csv"],
    [5, "05_content_feature_engineering.ipynb", "영화 메타데이터 기반 콘텐츠 피처 생성", "user_content_features.csv"],
    [6, "06_significance_tests.ipynb", "집단별 유의성 검정", "유의성 검정표"],
    [7, "07_modeling_baseline.ipynb", "1차 모델링과 성능 확인", "모델 성능표, feature importance"],
    [8, "08_xai_shap_interpretation.ipynb", "SHAP 기반 XAI 분석", "SHAP summary, dependence plot"],
    [9, "09_segmentation_strategy.ipynb", "세그먼트 정의와 전략 연결", "세그먼트별 전략표"],
    [10, "10_business_simulation_and_final_checks.ipynb", "비즈니스 시뮬레이션과 최종 검산", "기대효과표, 최종 숫자"],
], columns=["step", "notebook", "purpose", "main_output"])

notebook_plan

## 0-13. 현재 최선의 스토리라인

현재까지 가장 방어 가능한 스토리라인은 다음입니다.

```text
100원딜 고객은 일반 고객보다 재구독률이 낮다.
그러나 모든 100원딜 고객이 같은 방식으로 이탈하는 것은 아니다.

2인 요금제 고객은 TV 시청 가능성과 적정 가격대의 균형으로
생활형 이용에 정착할 가능성이 높고,
재구독률도 상대적으로 높다.

반면 4인 요금제 고객은 프로모션 기간에는 고사양 요금제를 체험하지만,
정가 전환 시 가격 부담이 커져 재구독률이 크게 낮다.

또한 단순 시청량보다 초반 루틴화와 후반 체험 소진 패턴이 더 중요하다.
콘텐츠 메타데이터는 이탈의 핵심 원인이라기보다,
세그먼트의 성격을 설명하는 보조 변수로 활용한다.
```

이 스토리라인은 이후 SHAP, 세그먼트, 리텐션 전략, 비즈니스 시뮬레이션으로 연결되어야 합니다.

## 0-14. 세그먼트 후보

현재 단계에서 확정이 아니라 후보로 둘 세그먼트는 다음입니다.

| 세그먼트 후보 | 정의 방향 | 현재 해석 |
|---|---|---|
| 100원딜 4인 고위험 체험형 | `is_100won=1`, `max_screen=4` 중심 | 정가 전환 부담이 큰 고위험군 |
| 100원딜 2인 생활정착형 | `is_100won=1`, `max_screen=2` 중심 | TV 이용과 가격 균형이 맞는 안정군 |
| 100원딜 1인 모바일 탐색형 | `is_100won=1`, `max_screen=1` 중심 | 모바일 중심 탐색 고객 |
| 패밀리/생활형 보조 세그먼트 | 2인, 키즈/가족/전체관람가 콘텐츠 성향 | 2인 안정군을 설명하는 보조 축 |
| 프리미엄 액션 체험소진형 | 4인, 액션/장편/후반 증가 성향 | 4인 위험군을 설명하는 보조 축 |
| 일반 유입 안정형 | 비100원딜 고객 | 비교 기준 또는 안정 기준군 |

이 세그먼트들은 `09_segmentation_strategy.ipynb`에서 최종 확정해야 합니다.

## 0-15. 금지해야 할 표현과 안전한 표현

이 프로젝트는 해석을 잘못하면 인과 과장이나 p-hacking처럼 보일 수 있습니다. 따라서 표현을 조심해야 합니다.

| 피해야 할 표현 | 이유 | 안전한 표현 |
|---|---|---|
| 100원딜이 이탈을 유발했다 | 인과 단정 불가 | 100원딜 고객군은 일반 고객군과 다른 재구독 행동을 보였다 |
| 많이 보면 재구독한다 | 단순 시청량 변수 약함 | 시청 총량보다 이용 패턴이 더 중요해 보인다 |
| 특정 장르가 이탈을 결정한다 | 장르 단독 효과 약함 | 콘텐츠 메타데이터는 세그먼트 성격 설명에 활용한다 |
| 키즈/애니 고객은 재구독률이 유의하게 높다 | 가설 검정에서 유의하지 않음 | 패밀리 콘텐츠 성향은 생활형 이용 해석의 보조 가설이다 |
| 모델이 이탈 고객을 정확히 예측한다 | AUC 높지 않음 | 모델은 리스크 랭킹과 세그먼트 탐색에 활용한다 |
| 4주차 행동을 보면 이탈을 잘 맞힐 수 있다 | 대응기간 누수 위험 | 1~3주차 행동으로 4주차 대응 대상을 선별한다 |

## 0-16. 최종 산출물 계획

최종적으로 만들어야 할 산출물은 네 종류입니다.

### 1. 데이터 산출물

```text
membership_preprocessed.csv
view_history_observation_window.csv
movie_metadata_unified_v2.csv
user_usage_features.csv
user_content_features.csv
modeling_feature_table_with_content.csv
```

### 2. 분석 산출물

```text
significance_tests.csv
model_holdout_metrics.csv
feature_importance.csv
risk_decile_table.csv
segment_rate_table.csv
```

### 3. XAI 산출물

```text
shap_summary_plot.png
shap_dependence_max_screen.png
shap_dependence_week_pattern.png
shap_feature_importance.csv
```

### 4. 보고/전략 산출물

```text
segment_strategy_table.csv
business_simulation_table.csv
final_report_figures/
final_presentation_notes.md
```

## 0-17. 다음 작업

이 노트북 다음에 작성할 파일은 `01_data_overview.ipynb`입니다.

`01_data_overview.ipynb`에서 해야 할 일은 다음입니다.

1. 원본 데이터 파일 존재 여부 확인
2. 각 파일의 행 수와 컬럼 수 확인
3. 주요 키 구조 확인
4. `Membership`의 target 분포 확인
5. `is_promotion`, `price`, `max_screen` 기본 분포 확인
6. `User_Mapping`의 중복 구조 확인
7. `View_History`의 날짜 범위와 고유 유저, 고유 영화 수 확인
8. `Movie_Master`, Wavve, KOBIS의 기본 커버리지 확인
9. 이후 전처리 노트북에서 사용할 검산 기준 작성

00번은 설계 문서이고, 01번부터 실제 데이터 확인이 시작됩니다.

## 0-18. 이 노트북을 Git에 올릴 때의 권장 위치

현재 저장소 구조를 고려하면 이 노트북은 아래 위치에 두는 것을 권장합니다.

```text
park.ingyeom/notebooks/00_project_plan/00_project_plan.ipynb
```

기존 `notebooks` 폴더가 `01_data_check`, `02_preprocessing`처럼 단계별 하위 폴더를 가지고 있으므로, 00번도 같은 방식으로 별도 폴더를 두는 것이 자연스럽습니다.

추가로 Markdown 백업을 함께 두면 GitHub에서 diff를 보기가 더 쉽습니다.

```text
park.ingyeom/notebooks/00_project_plan/00_project_plan.md
```